# 使用 Autogen 实现智能体 RAG

本笔记本演示了如何使用 Autogen 智能体实现检索增强生成(RAG),并具有增强的评估功能。

SQLite 版本修复
如果您遇到以下错误:
```
RuntimeError: Your system has an unsupported version of sqlite3. Chroma requires sqlite3 >= 3.35.0
```

请在笔记本开头取消注释以下代码块:

In [2]:
# %pip install pysqlite3-binary
# __import__('pysqlite3')
# import sys
# sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
!pip install chromadb

In [3]:
import os
import time
import asyncio
from typing import List, Dict
from dotenv import load_dotenv

from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken
from autogen_agentchat.messages import TextMessage
from azure.core.credentials import AzureKeyCredential
from autogen_ext.models.azure import AzureAIChatCompletionClient

import chromadb

load_dotenv()

True

## 创建客户端

首先,我们初始化 Azure AI 聊天完成客户端。该客户端将用于与 Azure OpenAI 服务交互,以生成对用户查询的响应。

In [4]:
client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    credential=AzureKeyCredential(os.getenv("GITHUB_TOKEN")),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

## 向量数据库初始化

我们使用持久化存储初始化 ChromaDB,并添加增强的示例文档。ChromaDB 将用于存储和检索文档,为生成准确的响应提供上下文。

In [5]:
# 使用持久化存储初始化 ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.create_collection(
    name="travel_documents",
    metadata={"description": "travel_service"},
    get_or_create=True
)

# 增强的示例文档
documents = [
    "Contoso Travel 提供全球异国目的地的豪华度假套餐。",
    "我们的优质旅行服务包括个性化行程规划和 24/7 礼宾支持。",
    "Contoso 的旅行保险涵盖医疗紧急情况、行程取消和行李丢失。",
    "热门目的地包括马尔代夫、瑞士阿尔卑斯山和非洲野生动物园。",
    "Contoso Travel 提供精品酒店和私人导游的独家访问权限。"
]

# 添加带有元数据的文档
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "training", "type": "explanation"} for _ in documents]
)

## 上下文提供者实现

`ContextProvider` 类处理来自多个源的上下文检索和集成:

1. **向量数据库检索**: 使用 ChromaDB 对旅行文档执行语义搜索
2. **天气信息**: 维护主要城市的模拟天气数据库
3. **统一上下文**: 将文档和天气数据组合成综合上下文

关键方法:
- `get_retrieval_context()`: 根据查询检索相关文档
- `get_weather_data()`: 为指定位置提供天气信息
- `get_unified_context()`: 结合文档和天气上下文以增强响应

In [6]:
class ContextProvider:
    def __init__(self, collection):
        self.collection = collection
        # 模拟天气数据库
        self.weather_database = {
            "new york": {"temperature": 72, "condition": "Partly Cloudy", "humidity": 65, "wind": "10 mph"},
            "london": {"temperature": 60, "condition": "Rainy", "humidity": 80, "wind": "15 mph"},
            "tokyo": {"temperature": 75, "condition": "Sunny", "humidity": 50, "wind": "5 mph"},
            "sydney": {"temperature": 80, "condition": "Clear", "humidity": 45, "wind": "12 mph"},
            "paris": {"temperature": 68, "condition": "Cloudy", "humidity": 70, "wind": "8 mph"},
        }
    
    def get_retrieval_context(self, query: str) -> str:
        """根据查询从向量数据库中检索相关文档。"""
        results = self.collection.query(
            query_texts=[query],
            include=["documents", "metadatas"],
            n_results=2
        )
        context_strings = []
        if results and results.get("documents") and len(results["documents"][0]) > 0:
            for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                context_strings.append(f"Document: {doc}\nMetadata: {meta}")
        return "\n\n".join(context_strings) if context_strings else "No relevant documents found"
    
    def get_weather_data(self, location: str) -> str:
        """模拟检索给定位置的天气数据。"""
        if not location:
            return ""
            
        location_key = location.lower()
        if location_key in self.weather_database:
            data = self.weather_database[location_key]
            return f"Weather for {location.title()}:\n" \
                   f"Temperature: {data['temperature']}°F\n" \
                   f"Condition: {data['condition']}\n" \
                   f"Humidity: {data['humidity']}%\n" \
                   f"Wind: {data['wind']}"
        else:
            return f"No weather data available for {location}."
    
    def get_unified_context(self, query: str, location: str = None) -> str:
        """返回统一上下文,结合文档检索和天气数据。"""
        retrieval_context = self.get_retrieval_context(query)
        
        weather_context = ""
        if location:
            weather_context = self.get_weather_data(location)
            weather_intro = f"\nWeather Information for {location}:\n"
        else:
            weather_intro = ""
        
        return f"Retrieved Context:\n{retrieval_context}\n\n{weather_intro}{weather_context}"

## 智能体配置

我们配置检索和助手智能体。检索智能体专门使用语义搜索查找相关信息,而助手根据检索到的信息生成详细响应。

In [7]:
# 创建具有增强功能的智能体
assistant = AssistantAgent(
    name="assistant",
    model_client=client,
    system_message=(
        "You are a helpful AI assistant that provides answers using ONLY the provided context. "
        "Do NOT include any external information. Base your answer entirely on the context given below."
    ),
)

## 使用 RAG 处理查询

我们定义 `ask_rag` 函数来向助手发送查询、处理响应并评估它。该函数处理与助手的交互,并使用评估器来衡量响应的质量。

In [8]:
async def ask_rag_agent(query: str, context_provider: ContextProvider, location: str = None):
    """
    向助手智能体发送带有提供者上下文的查询。
    
    Args:
        query: 用户的问题
        context_provider: 上下文提供者实例
        location: 天气查询的可选位置
    """
    try:
        # 获取统一上下文
        context = context_provider.get_unified_context(query, location)
        
        # 使用上下文增强查询
        augmented_query = (
            f"{context}\n\n"
            f"User Query: {query}\n\n"
            "Based ONLY on the above context, please provide a helpful answer."
        )

        # 将增强的查询发送给助手
        start_time = time.time()
        response = await assistant.on_messages(
            [TextMessage(content=augmented_query, source="user")],
            cancellation_token=CancellationToken(),
        )
        processing_time = time.time() - start_time
        
        return {
            'query': query,
            'response': response.chat_message.content,
            'processing_time': processing_time,
            'location': location
        }
    except Exception as e:
        print(f"Error processing query: {e}")
        return None

# 示例用法

我们初始化评估器并定义要处理和评估的查询。

In [ ]:
async def main():
    # 初始化上下文提供者
    context_provider = ContextProvider(collection)
    
    # 示例查询
    queries = [
        {"query": "What does Contoso's travel insurance cover?"},
        {"query": "What's the weather like in London?", "location": "london"},
        {"query": "What luxury destinations does Contoso offer and what's the weather in Paris?", "location": "paris"},
    ]
    
    print("=== Autogen RAG Demo ===")
    for query_data in queries:
        query = query_data["query"]
        location = query_data.get("location")
        
        print(f"\n\nQuery: {query}")
        if location:
            print(f"Location: {location}")
        
        # 显示正在使用的上下文
        context = context_provider.get_unified_context(query, location)
        print("\n--- Context Used ---")
        print(context)
        print("-------------------")
        
        # 从智能体获取响应
        result = await ask_rag_agent(query, context_provider, location)
        if result:
            print(f"\nResponse: {result['response']}")
        print("\n" + "="*50)

## 运行脚本

我们检查脚本是在交互环境中运行还是在标准脚本中运行,并相应地运行主函数。

In [12]:
if __name__ == "__main__":
    if asyncio.get_event_loop().is_running():
        await main()
    else:
        asyncio.run(main())

=== Autogen RAG Demo ===


Query: What does Contoso's travel insurance cover?

--- Context Used ---
Retrieved Context:
Document: Contoso's travel insurance covers medical emergencies, trip cancellations, and lost baggage.
Metadata: {'source': 'training', 'type': 'explanation'}

Document: Contoso Travel offers luxury vacation packages to exotic destinations worldwide.
Metadata: {'source': 'training', 'type': 'explanation'}


-------------------

Response: Contoso's travel insurance covers medical emergencies, trip cancellations, and lost baggage.



Query: What's the weather like in London?
Location: london

--- Context Used ---
Retrieved Context:
Document: Popular destinations include the Maldives, Swiss Alps, and African safaris.
Metadata: {'source': 'training', 'type': 'explanation'}

Document: Contoso Travel offers luxury vacation packages to exotic destinations worldwide.
Metadata: {'source': 'training', 'type': 'explanation'}


Weather Information for london:
Weather for London:
T